In [1]:
import json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical

SEED, MAX_EPOCHS, PATIENCE, BATCH = 42, 40, 5, 256
RESULTS, MODELS = Path("results"), Path("models")
RESULTS.mkdir(exist_ok=True); MODELS.mkdir(exist_ok=True)
CLASSES = ["nothing", "one_pair", "two_pairs", "three_of_a_kind", "straight",
           "flush", "full_house", "four_of_a_kind", "straight_flush", "royal_flush"]

d = np.load("poker_prepared.npz")                      # made by the data-preparation cells
X_train, y_train, X_val, y_val, X_test, y_test = (d[k] for k in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test"])
Y_train, Y_val, Y_test = (to_categorical(a, 10) for a in (y_train, y_val, y_test))

def build_model(hidden, activation="relu", dropout=0.0):
    """hidden = list of layer sizes, e.g. [128, 128]; dropout is applied after every hidden layer."""
    stack = [keras.Input((X_train.shape[1],))]
    for units in hidden:
        stack.append(layers.Dense(units, activation=activation))
        if dropout:
            stack.append(layers.Dropout(dropout))
    stack.append(layers.Dense(10, activation="softmax"))        # one probability per hand type
    model = keras.Sequential(stack)
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model

In [2]:
def run_experiment(name, hidden, activation="relu", dropout=0.0, force=False):
    path = RESULTS / f"{name}.json"
    if path.exists() and not force:                              # finished runs are never repeated
        return json.loads(path.read_text())
    keras.utils.set_random_seed(SEED)
    model = build_model(hidden, activation, dropout)
    stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)
    t = time.perf_counter()
    h = model.fit(X_train, Y_train, validation_data=(X_val, Y_val), epochs=MAX_EPOCHS,
                  batch_size=BATCH, callbacks=[stop], verbose=0)
    seconds = time.perf_counter() - t
    test_loss, test_acc = model.evaluate(X_test, Y_test, batch_size=4096, verbose=0)
    pred = model.predict(X_test, batch_size=4096, verbose=0).argmax(1)
    recall = [float((pred[y_test == c] == c).mean()) for c in range(10)]    # per-class accuracy on the test set
    hist = {k: [float(v) for v in vals] for k, vals in h.history.items()}
    best = int(np.argmin(hist["val_loss"]))
    result = dict(name=name, hidden=hidden, activation=activation, dropout=dropout,
                  epochs_run=len(hist["loss"]), best_epoch=best + 1, seconds=round(seconds, 1),
                  val_acc=hist["val_accuracy"][best], val_loss=hist["val_loss"][best],
                  test_acc=float(test_acc), test_loss=float(test_loss),
                  recall=recall, mean_recall=float(np.mean(recall)), history=hist)
    model.save(MODELS / f"{name}.keras")
    path.write_text(json.dumps(result))
    print(f"{name:24s} epochs={result['epochs_run']:2d}  {seconds:5.0f}s  val_acc={result['val_acc']:.4f}  test_acc={test_acc:.4f}")
    return result

In [3]:
ARCH = {"1x32": [32], "1x128": [128], "2x64": [64, 64], "2x128": [128, 128],
        "3x128": [128, 128, 128], "3x256": [256, 256, 256]}
EXPERIMENTS  = [(f"{a}_relu_d0.0", h, "relu", 0.0) for a, h in ARCH.items()]                     # sizes
EXPERIMENTS += [(f"{a}_{act}_d0.0", ARCH[a], act, 0.0) for a in ("1x32", "2x128") for act in ("sigmoid", "tanh")]  # activations
EXPERIMENTS += [(f"{a}_relu_d{p}", ARCH[a], "relu", p) for a in ("2x128", "3x256") for p in (0.2, 0.5)]            # dropout
print(len(EXPERIMENTS), "experiments")

for exp in EXPERIMENTS:          # slow part: safe to stop and restart, finished runs are skipped
    run_experiment(*exp)

14 experiments
1x32_relu_d0.0           epochs=40     84s  val_acc=0.7826  test_acc=0.7828
1x128_relu_d0.0          epochs=40     82s  val_acc=0.9978  test_acc=0.9980
2x64_relu_d0.0           epochs=30     75s  val_acc=0.9959  test_acc=0.9961
2x128_relu_d0.0          epochs=40    153s  val_acc=0.9999  test_acc=0.9999
3x128_relu_d0.0          epochs=40    121s  val_acc=0.9999  test_acc=0.9998
3x256_relu_d0.0          epochs=40    147s  val_acc=0.9999  test_acc=0.9999
1x32_sigmoid_d0.0        epochs=40     51s  val_acc=0.7252  test_acc=0.7264
1x32_tanh_d0.0           epochs=40     59s  val_acc=0.8926  test_acc=0.8938
2x128_sigmoid_d0.0       epochs=40    102s  val_acc=0.9999  test_acc=0.9999
2x128_tanh_d0.0          epochs=40     94s  val_acc=1.0000  test_acc=1.0000
2x128_relu_d0.2          epochs=40    109s  val_acc=0.9974  test_acc=0.9973
2x128_relu_d0.5          epochs=40    107s  val_acc=0.9402  test_acc=0.9399
3x256_relu_d0.2          epochs=21    231s  val_acc=0.9995  test_acc=0.99

In [ ]:
def summary():
    rows = [json.loads(p.read_text()) for p in RESULTS.glob("*.json")]
    df = pd.DataFrame(rows).drop(columns=["history", "recall"])
    df["hidden"] = df["hidden"].astype(str)
    return df.sort_values(["val_acc", "val_loss"], ascending=[False, True]).reset_index(drop=True)

def plot_run(name):
    r = json.loads((RESULTS / f"{name}.json").read_text()); h = r["history"]; ep = range(1, len(h["loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
    for a, key, title in ((ax[0], "accuracy", "accuracy"), (ax[1], "loss", "cross-entropy loss")):
        a.plot(ep, h[key], label="train"); a.plot(ep, h["val_" + key], label="validation")
        a.set_xlabel("epoch"); a.set_title(title); a.legend()
    fig.suptitle(f"{name}   (test accuracy {r['test_acc']:.4f})"); fig.tight_layout(); plt.show()

def per_class(name):
    r = json.loads((RESULTS / f"{name}.json").read_text())
    return pd.Series(r["recall"], index=CLASSES, name=name).round(3)

summary()

In [ ]:
from sklearn.model_selection import StratifiedKFold

def cross_validate(hidden, activation="relu", dropout=0.0, k=5):
    """Balance only inside each fold's training part, so duplicated rare hands never leak into the fold being scored."""
    scores = []
    for fold, (tr, va) in enumerate(StratifiedKFold(k, shuffle=True, random_state=SEED).split(X_tr, y_tr), 1):
        Xb, yb = balance(X_tr[tr], y_tr[tr])
        keras.utils.set_random_seed(SEED + fold)
        model = build_model(hidden, activation, dropout)
        stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)
        model.fit(Xb, to_categorical(yb, 10), validation_data=(X_tr[va], to_categorical(y_tr[va], 10)),
                  epochs=MAX_EPOCHS, batch_size=BATCH, callbacks=[stop], verbose=0)
        pred = model.predict(X_tr[va], batch_size=4096, verbose=0).argmax(1)
        scores.append(float((pred == y_tr[va]).mean()))
        print(f"fold {fold}: accuracy {scores[-1]:.4f}")
    print(f"mean {np.mean(scores):.4f} +/- {np.std(scores):.4f}")
    return scores

In [ ]:
best_name = summary().loc[0, "name"]                           # highest validation accuracy (ties: lowest validation loss)
keras.models.load_model(MODELS / f"{best_name}.keras").save("best_poker_model.keras")
print("exported:", best_name)

In [ ]:
loaded = keras.models.load_model("best_poker_model.keras")
data = np.load("poker_prepared.npz")                           # dataset loaded from file
pred = loaded.predict(data["X_test"], batch_size=4096, verbose=0).argmax(1)
print("accuracy of the loaded model on the test file: %.4f" % (pred == data["y_test"]).mean())